# 04 — Modelo Predictivo: Holt-Winters

Entrena un modelo Holt-Winters (suavizamiento exponencial triple) sobre la serie temporal de tasa de desocupacion nacional y genera un pronostico a 6 meses con intervalo de confianza del 90%.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..') if Path('../data/03_primary').exists() else Path('.')
DATA_PRIMARY = ROOT / 'data' / '03_primary'
DATA_OUTPUT  = ROOT / 'data' / '04_model_output'
MODELS_DIR   = ROOT / 'models'
FIGURES_DIR  = ROOT / 'reports' / 'figures'

if not DATA_PRIMARY.exists():
    DATA_PRIMARY = ROOT / 'data' / 'processed'
    DATA_OUTPUT  = ROOT / 'data' / 'processed'

for d in [DATA_OUTPUT, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'DATA_PRIMARY : {DATA_PRIMARY}')
print(f'DATA_OUTPUT  : {DATA_OUTPUT}')
print(f'MODELS_DIR   : {MODELS_DIR}')

In [ ]:
csv_path = DATA_PRIMARY / 'serie_temporal_td.csv'
df = pd.read_csv(csv_path, parse_dates=['fecha'])
serie = df.set_index('fecha')['tasa_desocupacion'].sort_index().dropna()

print(f'Periodos: {len(serie)}  |  {serie.index.min().date()} -> {serie.index.max().date()}')
serie.tail()

In [ ]:
split = int(len(serie) * 0.80)
train, test = serie.iloc[:split], serie.iloc[split:]
print(f'Train: {len(train)} | Test: {len(test)}')

plt.figure(figsize=(10, 4))
train.plot(label='Train', color='steelblue')
test.plot(label='Test', color='orange')
plt.title('Tasa de Desocupacion Nacional - Division Train/Test')
plt.ylabel('Tasa (%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def train_holtwinters(s):
    return ExponentialSmoothing(
        s, trend='add', seasonal='add', seasonal_periods=12
    ).fit(optimized=True, use_brute=True)

hw_model = train_holtwinters(train)
p = hw_model.params
print('Modelo entrenado.')
print(f'  alpha={p["smoothing_level"]:.4f}  beta={p["smoothing_trend"]:.4f}  gamma={p["smoothing_seasonal"]:.4f}')

In [ ]:
pred_test = hw_model.forecast(len(test))
pred_test.index = test.index

mae  = float(np.mean(np.abs(test.values - pred_test.values)))
rmse = float(np.sqrt(np.mean((test.values - pred_test.values) ** 2)))
print(f'MAE  = {mae:.4f} pp')
print(f'RMSE = {rmse:.4f} pp')

In [ ]:
HORIZON = 6
Z_90 = 1.645

model_final = train_holtwinters(serie)
forecast_vals = model_final.forecast(HORIZON)

last_date = serie.index[-1]
freq = pd.infer_freq(serie.index) or 'MS'
future_idx = pd.date_range(start=last_date, periods=HORIZON + 1, freq=freq)[1:]

forecast_df = pd.DataFrame({
    'fecha'     : future_idx,
    'prediccion': forecast_vals.values,
    'ic_lower'  : forecast_vals.values - Z_90 * mae,
    'ic_upper'  : forecast_vals.values + Z_90 * mae,
    'mae'       : mae,
    'rmse'      : rmse,
})
forecast_df

In [ ]:
out_csv = DATA_OUTPUT / 'prediccion_td.csv'
forecast_df.to_csv(out_csv, index=False)
print(f'Predicciones: {out_csv}')

model_path = MODELS_DIR / 'holt_winters_td.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_final, f)
print(f'Modelo:       {model_path}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
train.plot(ax=ax, label='Train', color='steelblue', linewidth=1.5)
test.plot(ax=ax, label='Test (real)', color='orange', linewidth=1.5)
pred_test.plot(ax=ax, label='Prediccion test', color='orange', linestyle='--', linewidth=1.5)
ax.plot(forecast_df['fecha'], forecast_df['prediccion'], color='green', linewidth=2, label='Pronostico 6 meses')
ax.fill_between(forecast_df['fecha'], forecast_df['ic_lower'], forecast_df['ic_upper'], color='green', alpha=0.15, label='IC 90%')
ax.set_title(f'Pronostico Tasa de Desocupacion - Holt-Winters  (MAE={mae:.2f} pp, RMSE={rmse:.2f} pp)')
ax.set_ylabel('Tasa de Desocupacion (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig_path = FIGURES_DIR / 'forecast_td.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Grafica: {fig_path}')
plt.show()

In [ ]:
print('=' * 50)
print('  RESULTADOS HOLT-WINTERS')
print('=' * 50)
print(f'  Train / Test          : {len(train)} / {len(test)}')
print(f'  MAE                   : {mae:.4f} pp')
print(f'  RMSE                  : {rmse:.4f} pp')
print(f'  Horizonte             : {HORIZON} meses')
print(f'  Intervalo de confianza: 90%')
print('=' * 50)
print()
print(forecast_df[['fecha','prediccion','ic_lower','ic_upper']].to_string(index=False))